In [1]:
import os
import numpy as np
import pickle
import matplotlib.pyplot as plt

from time import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from collections import OrderedDict

# 한글 폰트 및 마이너스 기호 표시 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [3]:
torch.cuda.init()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.reset_peak_memory_stats(device=None)
print("현재 디바이스:", device)

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

현재 디바이스: cuda


In [4]:
# CIFAR-100 데이터 로드 함수
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

data_path = './cifar-100-python'
# 데이터 경로 설정
train_path = os.path.join(data_path, 'train')
test_path = os.path.join(data_path, 'test')

# 학습 데이터 로드
train_data = unpickle(train_path)
test_data = unpickle(test_path)

# 메타데이터 로드 (클래스 이름 등)
meta_path = os.path.join(data_path, 'meta')
meta_data = unpickle(meta_path)

fine_label_names = [name.decode() for name in meta_data[b'fine_label_names']]
coarse_label_names = [name.decode() for name in meta_data[b'coarse_label_names']]

# 데이터 구조 확인
print("학습 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in train_data.keys()])
print("테스트 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in test_data.keys()])
print("메타 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in meta_data.keys()])

학습 데이터 키: ['filenames', 'batch_label', 'fine_labels', 'coarse_labels', 'data']
테스트 데이터 키: ['filenames', 'batch_label', 'fine_labels', 'coarse_labels', 'data']
메타 데이터 키: ['fine_label_names', 'coarse_label_names']


In [5]:
def one_hot_encode(y, num_classes):
    return np.eye(num_classes)[y.astype(int)]

# train set 으로 validation set 분할
x = train_data[b'data']
x = x.reshape(-1, 3, 32, 32)
t = np.array(train_data[b'coarse_labels'])
x_train, x_val, y_train, y_val = train_test_split(x, t, test_size=0.2, random_state=42, stratify=t)

# test set 정의
x_test = test_data[b'data']
x_test = x_test.reshape(-1, 3, 32, 32)
y_test = np.array(test_data[b'coarse_labels'])

x_test = x_test.astype(np.float32) / 255.0
x_train = x_train.astype(np.float32) / 255.0
x_val = x_val.astype(np.float32) / 255.0

y_train = one_hot_encode(y_train, 20)
y_val = one_hot_encode(y_val, 20)
y_test = one_hot_encode(y_test, 20)

x_train.shape, x_val.shape, x_test.shape, y_train.shape, y_val.shape, y_test.shape

((40000, 3, 32, 32),
 (10000, 3, 32, 32),
 (10000, 3, 32, 32),
 (40000, 20),
 (10000, 20),
 (10000, 20))

In [6]:
class CIFAR100Dataset(Dataset):
    def __init__(self, images, labels, transform=None): # transform 인자 추가
        self.images = images
        self.labels = labels
        self.transform = transform # transform 저장

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx] # (C, H, W) 형태의 NumPy 배열
        label = self.labels[idx]
        
        # NumPy 배열을 PyTorch 텐서로 변환
        # 이미지는 이미 float32로 정규화되어 있음
        image_tensor = torch.tensor(image) 
        label_tensor = torch.tensor(label) # 레이블은 원-핫 인코딩된 상태
        
        if self.transform:
            image_tensor = self.transform(image_tensor) # 변환 적용
            
        return image_tensor, label_tensor

In [7]:
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),  # 이미지 주위에 4픽셀 패딩 후 32x32 랜덤 크롭
    transforms.RandomHorizontalFlip(p=0.5), # 50% 확률로 좌우 반전
    # 필요한 경우 다른 변환 추가 가능 (예: transforms.ColorJitter)
])
train_dataset = CIFAR100Dataset(x_train, y_train, transform=train_transforms)
val_dataset = CIFAR100Dataset(x_val, y_val)
test_dataset = CIFAR100Dataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [8]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Convolutional Block 1 (Inspired by Keras example)
        self.conv1_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1) # padding='same'
        self.relu1_1 = nn.ReLU()
        self.conv1_2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1) # padding='same'
        self.relu1_2 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # (32, 16, 16)

        # Convolutional Block 2 (Inspired by Keras example)
        self.conv2_1 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1) # padding='same'
        self.relu2_1 = nn.ReLU()
        self.conv2_2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1) # padding='same'
        self.relu2_2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # (128, 8, 8)
        
        # Note: Keras example has one more Conv2D(128, (3,3)) after pooling, 
        # but for 20 classes and to keep it slightly simpler, we'll go to FC layers.
        # If needed, that layer can be added:
        # self.conv3_1 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        # self.relu3_1 = nn.ReLU()
        # self.flattened_size = 128 * 8 * 8 (if conv3_1 is added and no pooling after)

        self.flattened_size = 128 * 8 * 8 # 8192
        
        # Dense Block 1
        self.fc1 = nn.Linear(self.flattened_size, 256)
        self.relu_fc1 = nn.ReLU()
        self.bn_fc1 = nn.BatchNorm1d(256)
        self.drop_fc1 = nn.Dropout(p=0.3)

        # Dense Block 2
        self.fc2 = nn.Linear(256, 256)
        self.relu_fc2 = nn.ReLU()
        self.bn_fc2 = nn.BatchNorm1d(256)
        self.drop_fc2 = nn.Dropout(p=0.3)

        # Output Layer (for 20 coarse labels)
        self.fc_out = nn.Linear(256, 20)

    def forward(self, x):
        # Conv Block 1
        x = self.relu1_1(self.conv1_1(x))
        x = self.relu1_2(self.conv1_2(x))
        x = self.pool1(x)
        
        # Conv Block 2
        x = self.relu2_1(self.conv2_1(x))
        x = self.relu2_2(self.conv2_2(x))
        x = self.pool2(x)
        
        # Flatten
        x = x.view(-1, self.flattened_size)
        
        # Dense Block 1
        x = self.fc1(x)
        x = self.relu_fc1(x)
        x = self.bn_fc1(x)
        x = self.drop_fc1(x)
        
        # Dense Block 2
        x = self.fc2(x)
        x = self.relu_fc2(x)
        x = self.bn_fc2(x)
        x = self.drop_fc2(x)
        
        # Output Layer
        x = self.fc_out(x)
        return x

In [13]:
class SEBlock(nn.Module):
    """
    Squeeze-and-Excitation Block.
    입력 채널을 받아, 채널별 중요도를 학습하여 원래 특성에 적용합니다.
    """
    def __init__(self, in_channels, reduced_dim):
        super(SEBlock, self).__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # Squeeze: Global Average Pooling
            nn.Conv2d(in_channels, reduced_dim, kernel_size=1),  # Excitation: Fully Connected Layer (as 1x1 Conv)
            nn.SiLU(),  # Swish/SiLU 활성화 함수 (EfficientNet에서 사용)
            nn.Conv2d(reduced_dim, in_channels, kernel_size=1),  # Excitation: Fully Connected Layer
            nn.Sigmoid()  # 채널별 가중치를 0과 1 사이로 조정
        )

    def forward(self, x):
        se_weight = self.se(x)
        return x * se_weight # 원래 특성에 채널별 가중치 적용

In [14]:
class MBConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio, se_ratio=0.25, drop_connect_rate=0.2):
        super(MBConvBlock, self).__init__()
        self.stride = stride
        self.use_skip_connection = (stride == 1 and in_channels == out_channels) # 스트라이드가 1이고 입출력 채널이 같을 때만 skip connection 사용

        expanded_channels = in_channels * expand_ratio
        
        # Expansion phase (1x1 Convolution)
        self.expand_conv = nn.Conv2d(in_channels, expanded_channels, kernel_size=1, bias=False)
        self.bn0 = nn.BatchNorm2d(expanded_channels)
        self.silu0 = nn.SiLU()

        # Depthwise convolution
        self.depthwise_conv = nn.Conv2d(expanded_channels, expanded_channels, kernel_size=kernel_size, 
                                        stride=stride, padding=kernel_size//2, groups=expanded_channels, bias=False)
        self.bn1 = nn.BatchNorm2d(expanded_channels)
        self.silu1 = nn.SiLU()

        # Squeeze-and-Excitation block
        # reduced_dim은 expanded_channels의 se_ratio 배이지만, 최소값을 1로 설정하기도 함
        reduced_dim = max(1, int(expanded_channels * se_ratio)) 
        self.se_block = SEBlock(expanded_channels, reduced_dim=reduced_dim) # SEBlock은 이전에 정의된 것을 사용

        # Projection phase (1x1 Convolution)
        self.project_conv = nn.Conv2d(expanded_channels, out_channels, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # DropConnect (Stochastic Depth의 한 종류) - 여기서는 간단히 Dropout으로 대체하거나 생략 가능
        # EfficientNet에서는 DropConnect를 사용합니다. 간단하게는 nn.Dropout2d를 사용할 수 있습니다.
        self.drop_connect_rate = drop_connect_rate
        if self.use_skip_connection and self.drop_connect_rate > 0:
            self.dropout = nn.Dropout2d(p=drop_connect_rate) # 논문에서는 이미지 배치 단위로 적용

    def forward(self, x):
        identity = x

        out = self.expand_conv(x)
        out = self.bn0(out)
        out = self.silu0(out)

        out = self.depthwise_conv(out)
        out = self.bn1(out)
        out = self.silu1(out)

        out = self.se_block(out)

        out = self.project_conv(out)
        out = self.bn2(out)

        if self.use_skip_connection:
            if self.training and self.drop_connect_rate > 0:
                # DropConnect: 학습 시에만 적용, identity를 드롭아웃
                # 실제 DropConnect는 identity를 확률적으로 0으로 만들고 스케일링합니다.
                # 여기서는 간단히 출력에 dropout을 적용하거나, identity를 드롭하는 형태로 구현할 수 있습니다.
                # 좀 더 정확한 구현은 StochasticDepth를 참고해야 합니다.
                # out = self.dropout(out) # 간단한 예시
                # identity를 드롭하는 경우:
                if torch.rand(1).item() < self.drop_connect_rate and self.training:
                     out = out # identity를 더하지 않음 (또는 0을 더함)
                else:
                     out += identity
            else:
                out += identity
        return out

class EfficientNetCNN(nn.Module):
    def __init__(self, num_classes=20, dropout_rate=0.2):
        super(EfficientNetCNN, self).__init__()
        self.upsample = nn.Upsample(size=(224, 224), mode='bicubic', align_corners=False)

        # Stem
        out_channels_stem = 32
        self.stem_conv = nn.Conv2d(3, out_channels_stem, kernel_size=3, stride=2, padding=1, bias=False)
        self.stem_bn = nn.BatchNorm2d(out_channels_stem)
        self.stem_silu = nn.SiLU()

        # MBConv 블록 구성 (EfficientNet-B0의 stage별 파라미터 참고)
        # 예시: Stage 2 (1 MBConv block)
        self.mbconv1 = MBConvBlock(in_channels=32, out_channels=16, kernel_size=3, stride=1, expand_ratio=1)
        
        # 예시: Stage 3 (2 MBConv blocks)
        self.mbconv2_1 = MBConvBlock(in_channels=16, out_channels=24, kernel_size=3, stride=2, expand_ratio=6)
        self.mbconv2_2 = MBConvBlock(in_channels=24, out_channels=24, kernel_size=3, stride=1, expand_ratio=6)

        # ... (더 많은 MBConv 블록들을 EfficientNet 구조에 맞게 추가) ...
        # 현재는 mbconv2_2의 out_channels=24를 head_conv의 in_channels로 사용합니다.
        # 실제 EfficientNet-B0는 마지막 stage에서 320 채널을 출력하고, head_conv에서 1280 채널로 확장합니다.
        # 여기서는 단순화를 위해 마지막 블록의 출력을 바로 head로 연결합니다.
        # 따라서 head_conv의 in_channels를 마지막 MBConv 블록의 out_channels로 맞춰야 합니다.
        
        # Head Convolution 및 Classifier
        # 마지막 MBConv 블록의 출력 채널을 받음 (여기서는 self.mbconv2_2의 out_channels = 24)
        # EfficientNet-B0는 head에서 1280 채널 사용
        # 이 부분을 실제 EfficientNet 구조에 맞게 수정해야 합니다.
        # 예를 들어, 마지막 MBConv 블록의 출력이 24라면, head_conv의 입력도 24가 되어야 합니다.
        # 또는, EfficientNet 논문처럼 여러 MBConv stage를 거쳐 최종 채널 수를 맞추고 head로 연결해야 합니다.
        
        # 임시로 마지막 블록의 출력을 24로 가정하고 head 구성
        last_block_out_channels = 24 # 이 값은 실제 마지막 MBConv 블록의 out_channels와 일치해야 함
        head_in_channels = last_block_out_channels 
        head_out_channels = 1280 # EfficientNet-B0 head 기준
        
        self.head_conv = nn.Conv2d(head_in_channels, head_out_channels, kernel_size=1, bias=False)
        self.head_bn = nn.BatchNorm2d(head_out_channels)
        self.head_silu = nn.SiLU()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(head_out_channels, num_classes)

    def forward(self, x):
        x = self.upsample(x)
        x = self.stem_silu(self.stem_bn(self.stem_conv(x)))

        x = self.mbconv1(x)
        x = self.mbconv2_1(x)
        x = self.mbconv2_2(x)
        # ... (추가된 MBConv 블록들 통과) ...

        x = self.head_silu(self.head_bn(self.head_conv(x)))
        x = self.avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# ...existing code...

In [15]:
model = EfficientNetCNN(num_classes=20).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [16]:
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        target_labels = torch.max(labels, 1)[1] 
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, target_labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted_train = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted_train == target_labels).sum().item()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = 100 * correct_train / total_train
    
    # --- 검증 단계 ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            target_labels = torch.max(labels, 1)[1]
            
            outputs = model(images)
            loss = criterion(outputs, target_labels)
            val_loss += loss.item() * images.size(0)
            
            _, predicted_val = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted_val == target_labels).sum().item()
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = 100 * correct_val / total_val
    
    print(f'Epoch [{epoch+1}/{num_epochs}]: '
          f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%, '
          f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')

# --- 최종 테스트 단계 ---
model.eval()
correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in test_loader: # test_loader 사용
        images, labels = images.to(device), labels.to(device)
        target_labels = torch.max(labels, 1)[1]
        
        outputs = model(images)
        _, predicted_test = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted_test == target_labels).sum().item()

final_test_acc = 100 * correct_test / total_test
print(f'Test Accuracy on {total_test} test images: {final_test_acc:.2f}%')

Epoch [1/50]: Train Loss: 2.4710, Train Acc: 23.11%, Val Loss: 2.3228, Val Acc: 26.98%
Epoch [2/50]: Train Loss: 2.1935, Train Acc: 31.36%, Val Loss: 2.1390, Val Acc: 32.79%
Epoch [3/50]: Train Loss: 2.0391, Train Acc: 35.86%, Val Loss: 2.0675, Val Acc: 36.00%
Epoch [4/50]: Train Loss: 1.9494, Train Acc: 38.40%, Val Loss: 1.9447, Val Acc: 38.51%
Epoch [5/50]: Train Loss: 1.8787, Train Acc: 40.42%, Val Loss: 1.9079, Val Acc: 40.11%
Epoch [6/50]: Train Loss: 1.8167, Train Acc: 42.26%, Val Loss: 1.7991, Val Acc: 43.13%
Epoch [7/50]: Train Loss: 1.7618, Train Acc: 44.19%, Val Loss: 1.8252, Val Acc: 42.74%
Epoch [8/50]: Train Loss: 1.7187, Train Acc: 45.32%, Val Loss: 1.7423, Val Acc: 44.86%
Epoch [9/50]: Train Loss: 1.6723, Train Acc: 46.87%, Val Loss: 1.7065, Val Acc: 45.58%
Epoch [10/50]: Train Loss: 1.6349, Train Acc: 47.98%, Val Loss: 1.6527, Val Acc: 47.83%
Epoch [11/50]: Train Loss: 1.6035, Train Acc: 48.95%, Val Loss: 1.6627, Val Acc: 47.59%
Epoch [12/50]: Train Loss: 1.5735, Train 